# TRIBE v2 runner — NeuroProfile

Runs Meta's TRIBE v2 brain-encoding model on a video, then derives the
functional-systems mapping the reducer depends on.

**Order:** setup (1–6) → transcription fix (7) → encode a video (8) → ICA (9–20).
Setup only needs re-running after a runtime reset.

Model output is `(n_segments, 20484)` ---> fsaverage5 cortical surface only;
subcortical was never released. The model already applies the −5 s hemodynamic
offset and z-scores its output, so never repeat either downstream.

In [ ]:
# --- SETUP 1/6 · remove conflicting packages --------------------------------
# Colab ships a torch build tribev2's CUDA extensions can't load (shows up as a
# libcudart.so.13 error). Clear everything so pip can't half-resolve the graph.
print("Uninstalling existing packages...")
!pip uninstall -y tribev2 neuralset neuraltrain torch torchaudio transformers
print("Uninstallation complete.")

In [ ]:
# --- SETUP 2/6 · install the matched CUDA 12.1 triple -----------------------
# These three versions must match exactly. The trap: torchvision 0.20.0 hard-pins
# torch==2.5.0, but tribev2 wants torch>=2.5.1 — hence 0.20.1, not 0.20.0.
# After this, never `pip install -U transformers` or --force-reinstall torch;
# both break CUDA and you'll be deleting the runtime to recover.
print("Installing matched torch/torchvision/torchaudio for CUDA 12.1...")
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
print("Torch install complete.")

#### SETUP 3/6 · Restart the runtime now

`Runtime` > `Restart runtime` (or `Ctrl+M .`). The new torch won't load until you do.
Then continue from the next cell — do **not** re-run cells 1–2.

In [ ]:
# --- SETUP 4/6 · install tribev2 --------------------------------------------
# Run AFTER the restart. torch 2.5.1 + torchvision 0.20.1 already satisfy
# tribev2's pins, so pip won't swap them back out.
print("Reinstalling tribev2...")
!pip install git+https://github.com/facebookresearch/tribev2.git
print("tribev2 reinstallation complete.")

In [ ]:
# --- SETUP 5/6 · Hugging Face login -----------------------------------------
# tribev2 pulls Llama-3.2-3B, which is GATED: you need an approved HF account
# and a token with read access. Without it, from_pretrained fails at download.
from huggingface_hub import login
login()

In [ ]:
# --- SETUP 6/6 · load the model ---------------------------------------------
# TribeModel is an exca config/trainer object, NOT the network. The actual
# network appears at `model._model` only AFTER the first predict() call — it's a
# pydantic private attr, so it won't show up in vars(model).
from tribev2 import TribeModel

model = TribeModel.from_pretrained("facebook/tribev2")

In [ ]:
# --- Transcription fix · run once per runtime, BEFORE get_events_dataframe ---
# get_events_dataframe shells out to whisperX, which needs NLTK's punkt_tab.
# Colab's egress proxy blocks the auto-download unless you opt in explicitly.
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
import nltk
for res in ["punkt_tab", "punkt"]:
    nltk.download(res, download_dir="/root/nltk_data")

import os.path as p
print("punkt_tab present:", p.exists("/root/nltk_data/tokenizers/punkt_tab/english"))
# want: punkt_tab present: True

In [ ]:
# --- Encode a video ----------------------------------------------------------
# get_events_dataframe extracts audio and transcribes it (whisperX). `events`
# holds the transcript (text, start, duration, sentence) — that's the transcript
# artifact. Pass exactly ONE of video_path / audio_path / text_path.
events = model.get_events_dataframe(video_path="/content/IMG_5611.mp4")
preds, segments = model.predict(events)

# preds: (n_kept_segments, 20484) cortical vertices, one row per second (TR=1.0).
# predict() DROPS empty segments, so row count != clip length in seconds — a 103 s
# clip returned 222/300. Always align rows by segments[i].start, never by index.
print("preds shape:", preds.shape)        # (time, 20484) cortical-only
print("num segments:", len(segments))     # matches preds.shape[0]
print("one segment:", segments[0])        # Segment(start, duration, ...)

In [ ]:
# --- ICA STEP 1 · locate the latent->cortex readout --------------------------
# The readout is the final layer mapping the model's 2048-dim latent onto cortex.
# Its weights are what we decompose. Scans named_modules (Linear.weight) and
# named_parameters (raw Parameters, e.g. SubjectLayers) for any shape touching
# 20484. Expect a hit at `predictor.weights` -> (1, 2048, 20484).
import torch

N_CORTEX = preds.shape[1]        # 20484
net = model._model               # Lightning BrainModule, populated after predict()
print("net type:", type(net).__name__)

hits = []
for n, m in net.named_modules():
    w = getattr(m, "weight", None)
    if isinstance(w, torch.Tensor) and N_CORTEX in tuple(w.shape):
        hits.append(("module.weight", n, tuple(w.shape), type(m).__name__))
for n, p in net.named_parameters():
    if N_CORTEX in tuple(p.shape):
        hits.append(("parameter", n, tuple(p.shape), ""))

print("N_CORTEX =", N_CORTEX)
for h in hits:
    print(h)
if not hits:
    # fallback: dump candidate submodules so we can locate the readout by hand
    print("No 20484 weight found. Dumping candidate submodules:")
    for name in ["predictor", "low_rank_head", "subject_embed"]:
        for n, m in net.named_modules():
            if n.endswith(name):
                print(n, "->", m)

### ICA STEP 2 — FastICA on the readout → 5 spatial maps over cortex

The readout weight is `(1, 2048, 20484)` = (subjects, latent, cortex). There's only
one subject head, and it *is* the released average/unseen-subject readout, so no
subject-averaging is needed — just take `[0]`.

Fit with **latent dims as samples and vertices as features**, so each of the 5
components comes out as a spatial map over the 20,484 cortical vertices.

ICA takes a set of signals that are all jambled together and from several hidden sources, and pull the individual soruces back out.

I am using this because what we are looking for is not cleanly given to use they are overlapping, (the 5 brain region). SO we use CIA to discover the separate components so I can then keep the ones that matter.

Basically, using ICA to untangle the neural signals into its independent parts, so I can isolate the real response from everything else that is mixed in. (the various responses)

https://scikit-learn.org/stable/modules/decomposition.html#ica

In [ ]:
import numpy as np
from sklearn.decomposition import FastICA

# (1, 2048, 20484) -> drop the subject axis
W = net.predictor.weights.detach().float().cpu().numpy()
print("readout weight:", W.shape)
W = W[0]                                                    # (2048, 20484)
assert W.shape[1] == N_CORTEX, W.shape
print("ICA input:", W.shape, "(samples=latent=%d, features=cortex=%d)" % W.shape)

# 5 independent spatial maps. A ConvergenceWarning here is tolerable — only a
# problem if the maps look like noise when you read the top regions in step 3b.
ica = FastICA(n_components=5, random_state=0, max_iter=2000, tol=1e-4, whiten="unit-variance")
ica.fit(W)                     # samples = 2048 latent dims, features = 20484 vertices
comps = ica.components_        # (5, 20484)
print("components:", comps.shape)

np.save("/content/ica_components_fsav5.npy", comps)
print("saved -> /content/ica_components_fsav5.npy")

In [ ]:
# --- Persist to Drive before disconnecting -----------------------------------
# /content is wiped on disconnect. Saving W + comps to Drive means the ICA steps
# can resume later WITHOUT reloading the model (the slow part).
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/neuroprofile"
os.makedirs(SAVE_DIR, exist_ok=True)
np.save(f"{SAVE_DIR}/readout_W.npy", net.predictor.weights.detach().float().cpu().numpy())  # (1,2048,20484)
np.save(f"{SAVE_DIR}/ica_components_fsav5.npy", comps)
print("persisted to", SAVE_DIR)

In [ ]:
# --- RESUME HERE (no model needed) -------------------------------------------
# Everything from step 3 on needs only `comps`. Start a fresh runtime, run this,
# then skip to the analysis cell below.
from google.colab import drive; drive.mount("/content/drive")
import numpy as np
SAVE_DIR = "/content/drive/MyDrive/neuroprofile"
comps = np.load(f"{SAVE_DIR}/ica_components_fsav5.npy")     # (5, 20484)
N_CORTEX = comps.shape[1]                                    # 20484
W = np.load(f"{SAVE_DIR}/readout_W.npy")[0]                  # only if re-running FastICA
print("reloaded comps:", comps.shape)

### ICA STEP 3a — Glasser (HCP-MMP1) parcellation as fsaverage5 labels

Turns vertex indices into named brain regions. figshare ships the atlas on *full*
fsaverage (163,842 verts/hemi); fsaverage5's vertices are exactly the first 10,242
of those (nested icosahedra), so slicing `[:10242]` is an exact downsample.

The download is unreliable from Colab — figshare redirects to a presigned S3 URL
that expires in 10 seconds and the egress proxy blows it (403). The `.annot` files
are committed in the repo at `ica/atlas/` and uploaded to Drive; the cells below
read them from there. Same logic as an importable module: `notebooks/fetch_glasser.py`.

In [ ]:
# --- One-time · move uploaded files into Drive -------------------------------
# Run after uploading fetch_glasser.py and the two .annot files to /content.
# Puts them where the cells below expect: neuroprofile/ and neuroprofile/atlas/.
import os
import shutil

os.makedirs(SAVE_DIR, exist_ok=True)
DRIVE_ATLAS_DIR = os.path.join(SAVE_DIR, "atlas")
os.makedirs(DRIVE_ATLAS_DIR, exist_ok=True)

files_to_move_to_base = [
    "/content/fetch_glasser.py"
]
files_to_move_to_atlas = [
    "/content/lh.HCP-MMP1.annot",
    "/content/rh.HCP-MMP1.annot",
    "/content/SHA256SUMS"
]

for file_path in files_to_move_to_base:
    if os.path.exists(file_path):
        destination_path = os.path.join(SAVE_DIR, os.path.basename(file_path))
        shutil.move(file_path, destination_path)
        print(f"Moved {file_path} to {destination_path}")
    else:
        print(f"File not found: {file_path}. Skipping move.")

for file_path in files_to_move_to_atlas:
    if os.path.exists(file_path):
        destination_path = os.path.join(DRIVE_ATLAS_DIR, os.path.basename(file_path))
        shutil.move(file_path, destination_path)
        print(f"Moved {file_path} to {destination_path}")
    else:
        print(f"File not found: {file_path}. Skipping move.")

print("All specified files moved to Google Drive.")

In [ ]:
# --- OPTIONAL · build labels via the repo module -----------------------------
# Superseded by the next cell, which inlines this and needs no upload. Kept
# because it's the same code path the reducer will use on the Mac.
# Requires fetch_glasser.py on Drive AND Drive mounted first; invalidate_caches()
# is needed because Python caches sys.path entries that didn't exist at insert.
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/neuroprofile")
importlib.invalidate_caches()
from fetch_glasser import build_labels

labels, id2name = build_labels(n_cortex=N_CORTEX)
print("labels:", labels.shape, "| example names:", id2name.get(1), id2name.get(1001))

In [ ]:
# --- ICA STEPS 3 + 3b + 3c · assign regions, then read the components --------
# Self-contained: loads comps + atlas from Drive, builds vertex->region labels,
# averages each ICA map within each Glasser region, and assigns every region to
# its strongest component. Then prints the evidence needed to NAME them.
#
# Read the output in three parts:
#   "per component"    lopsided counts = one component's scale is dominating the
#                      argmax (fix: z-score each row of comps first)
#   "sign-robustness"  ICA sign is arbitrary, so a region straddling a +/- edge can
#                      cancel to ~0 under a signed mean. This recomputes with
#                      mean(|loading|) and reports disagreement. Low % = solid.
#   "component N: [...]" the 12 top-loading regions — this is what you name from.
import os, collections
import numpy as np
import nibabel.freesurfer.io as fsio
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/neuroprofile"

comps = np.load(f"{ROOT}/ica_components_fsav5.npy")      # (5, 20484)
N_CORTEX = comps.shape[1]

# nested-icosahedron downsample: first 10242 verts/hemi ARE fsaverage5
lh_lab, _, lh_names = fsio.read_annot(f"{ROOT}/atlas/lh.HCP-MMP1.annot")
rh_lab, _, rh_names = fsio.read_annot(f"{ROOT}/atlas/rh.HCP-MMP1.annot")
N5, OFF = N_CORTEX // 2, 1000            # OFF keeps L/R region ids distinct
labels = np.concatenate([lh_lab[:N5], rh_lab[:N5] + OFF]).astype(np.int32)

def _hemi(nm, h):
    nm = nm.decode() if isinstance(nm, bytes) else nm
    return nm if nm.startswith(("L_", "R_")) else f"{h}_{nm}"
id2name = {i: _hemi(lh_names[i], "L") for i in range(len(lh_names))}
id2name.update({i + OFF: _hemi(rh_names[i], "R") for i in range(len(rh_names))})

# '???' is the unlabelled medial wall, not a region -> excluded. Expect 360.
region_ids  = [int(i) for i in np.unique(labels) if "???" not in id2name.get(int(i), "???")]
region_means     = np.array([comps[:, labels == i].mean(axis=1) for i in region_ids])
region_means_abs = np.array([np.abs(comps[:, labels == i]).mean(axis=1) for i in region_ids])
assigned, assigned_abs = np.abs(region_means).argmax(axis=1), region_means_abs.argmax(axis=1)

d = int((assigned != assigned_abs).sum())
print("regions:", len(region_ids), "(expect 360) | per component:",
      dict(collections.Counter(assigned.tolist())))
print(f"sign-robustness: {d}/{len(region_ids)} disagree ({100*d/len(region_ids):.1f}%)\n")
for c in range(comps.shape[0]):
    print(f"component {c}: {[id2name[region_ids[k]] for k in np.argsort(-np.abs(region_means[:, c]))[:12]]}\n")

In [ ]:
# --- ICA STEPS 4 + 5 + 6 · affect/reward, anchor check, FREEZE ---------------
# Names below were read off the previous cell's top-region lists.
# Writes the three artifacts the reducer depends on. Already run 2026-08-15 —
# re-running overwrites the frozen mapping and invalidates downstream numbers.
import json

SYSTEM_NAMES = {0: "audiovisual_integration", 1: "social_sts_tpj", 2: "visual_motion",
                3: "auditory", 4: "dmn_scene_medial_parietal"}
SYSTEM_TIERS = {0: "moderate", 1: "moderate", 2: "high", 3: "high", 4: "moderate"}
ASSIGNMENT = assigned            # -> assigned_abs if sign-robustness had been high

# STEP 4 · affect/reward is HAND-ADDED, not ICA-derived. TRIBE released no
# subcortex, so there's no amygdala/accumbens -> anterior insula + ACC + OFC/vmPFC
# as cortical proxies, tiered "low". PoI1/PoI2 (posterior insula) deliberately
# excluded: interoceptive/somatosensory, not affect.
AFFECT_REWARD = {"AVI","AAIC","MI","FOP4","FOP5","a24","a24pr","p24","p24pr","a32pr",
                 "p32","d32","s32","33pr","25","OFC","pOFC","10r","10v","10pp","11l",
                 "13l","47m","47s"}
def base_name(nm):               # 'L_AVI_ROI' -> 'AVI'
    if nm.startswith(("L_", "R_")): nm = nm[2:]
    return nm[:-4] if nm.endswith("_ROI") else nm

region_system = {id2name[r]: int(ASSIGNMENT[k]) for k, r in enumerate(region_ids)}
AFFECT_ID = comps.shape[0]
n_aff = 0
for r in list(region_system):
    if base_name(r) in AFFECT_REWARD:
        region_system[r], n_aff = AFFECT_ID, n_aff + 1
SYSTEM_NAMES[AFFECT_ID], SYSTEM_TIERS[AFFECT_ID] = "affect_reward", "low"
print(f"affect/reward: {n_aff} regions (expect 48)")

# STEP 6 · freeze. labels.npy matters as much as the map: without it the reducer
# has to re-download the atlas every run and golden tests aren't reproducible.
OUT = f"{ROOT}/ica"
os.makedirs(OUT, exist_ok=True)
json.dump({
    "n_cortex": int(N_CORTEX), "surface": "fsaverage5",
    "vertex_order": "LH 0:10242, RH 10242:20484",
    "parcellation": "HCP-MMP1 (Glasser 360), figshare 3498446, [:10242]/hemi -> fsaverage5",
    "derivation": "FastICA(n_components=5, random_state=0) on TRIBE v2 "
                  "model._model.predictor.weights[0] (2048 latent x 20484 cortex)",
    "assignment_variant": "signed",
    "sign_robustness_disagreement": "22/360 (6.1%)",
    "known_limitations": "Components 3 and 4 hold 101/104 of 360 regions and act as "
                         "catch-alls; V1 and M1 land there with weak loadings. Early "
                         "visual and motor assignments are LOW CONFIDENCE — consistent "
                         "with V-JEPA2 spatial averaging and no motor task in movies.",
    "systems": {str(i): {"name": SYSTEM_NAMES[i], "tier": SYSTEM_TIERS[i],
                         "derived": i != AFFECT_ID} for i in sorted(SYSTEM_NAMES)},
    "region_system": region_system,
}, open(f"{OUT}/region_system_map.json", "w"), indent=2, sort_keys=True)

np.save(f"{OUT}/fsaverage5_glasser_labels.npy", labels.astype(np.int16))
json.dump({"id2name": {str(k): v for k, v in id2name.items()}, "region_ids": region_ids},
          open(f"{OUT}/fsaverage5_glasser_ids.json", "w"), indent=2)

print("froze ->", OUT)
for k in sorted(SYSTEM_NAMES):
    n = sum(1 for v in region_system.values() if v == k)
    print(f"  {k} {SYSTEM_NAMES[k]:<28} {SYSTEM_TIERS[k]:<9} {n} regions")